# Análise Comparativa dos Dados: HFSS vs. Tidy3D vs. Analítico (FR-4/Ar e RO3010/Rutila)

Este notebook lê os arquivos de dados salvos em `FR4/Dados/` e `RO3010/Dados/` e constrói as matrizes comparativas entre os métodos **HFSS**, **Tidy3D** e **Analítico** para todos os modelos estudados:

1. **Branch-Line** — Frequência de ressonância de $S_{11}$ (GHz)
2. **Cavidade Ressonante** — Frequência do modo $\text{TE}_{101}$ (GHz) *(Ar $\varepsilon_r = 1{,}0$ e Rutila $\varepsilon_r = 82{,}0$)*
3. **Antena Patch** — Frequência de ressonância de $S_{11}$ (GHz)
4. **Divisor de Wilkinson** — Frequência de ressonância de $S_{11}$ (GHz)
5. **Linha de Transmissão (Microfita)** — Menor valor de $S_{11}$ (dB)


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

DIR_FR4 = Path("FR4/Dados")
DIR_RO3010 = Path("RO3010/Dados")


In [2]:
def freq_ressonancia_s11(pasta: Path, arquivo: str, branchline_analitico: bool = False) -> float:
    """Retorna a frequência (GHz) correspondente ao mínimo de S11 (dB)."""
    caminho = pasta / arquivo
    if branchline_analitico:
        cols = ["Freq_GHz", "S11_dB", "S21_dB", "S31_dB", "S41_dB", "Fase_S21", "Fase_S31", "Delta_Fase"]
        df = pd.read_csv(caminho, sep="\t", comment="#", names=cols)
    else:
        df = pd.read_csv(caminho, sep="\t")
    idx_min = df["S11_dB"].idxmin()
    return float(df.loc[idx_min, "Freq_GHz"])


def freq_cavidade_te101(pasta: Path, arquivo: str, col_modo: str, col_freq: str) -> float:
    """Retorna a frequência (GHz) do modo TE_101 na tabela de modos da cavidade."""
    df = pd.read_csv(pasta / arquivo, sep="\t")
    val = df.loc[df[col_modo] == "TE_101", col_freq].iloc[0]
    return float(val)


def min_db_linha_transmissao(pasta: Path, arquivo: str) -> float:
    """Retorna o menor valor de S11 (dB) da linha de transmissão."""
    df = pd.read_csv(pasta / arquivo, sep="\t")
    return float(df["S11_dB"].min())


def montar_matriz_substrato(pasta: Path, sufixo_sub: str, sufixo_cav: str) -> pd.DataFrame:
    dados = {
        "HFSS": [
            freq_ressonancia_s11(pasta, "branchline_3ghz_curvas_hfss.txt"),
            freq_cavidade_te101(pasta, "cavidade_modos_20_hfss.txt", "Modo_Teorico", "Freq_GHz_HFSS"),
            freq_ressonancia_s11(pasta, "patch_3ghz_curvas_hfss.txt"),
            freq_ressonancia_s11(pasta, "wilkinson_3ghz_curvas_hfss.txt"),
            min_db_linha_transmissao(pasta, "linha_microfita_curvas_hfss.txt"),
        ],
        "Tidy3D": [
            freq_ressonancia_s11(pasta, "branchline_3ghz_curvas_tidy3d.txt"),
            freq_cavidade_te101(pasta, "cavidade_modos_20_tidy3d.txt", "Modo_Teorico", "Freq_GHz_Tidy3D"),
            freq_ressonancia_s11(pasta, "patch_3ghz_curvas_tidy3d.txt"),
            freq_ressonancia_s11(pasta, "wilkinson_3ghz_curvas_tidy3d.txt"),
            min_db_linha_transmissao(pasta, "linha_microfita_curvas_tidy3d.txt"),
        ],
        "Analítico": [
            freq_ressonancia_s11(pasta, "branchline_3ghz_curvas.txt", branchline_analitico=True),
            freq_cavidade_te101(pasta, "cavidade_modos_30.txt", "Modo", "Freq_GHz"),
            freq_ressonancia_s11(pasta, "patch_3ghz_curvas.txt"),
            freq_ressonancia_s11(pasta, "wilkinson_3ghz_curvas.txt"),
            min_db_linha_transmissao(pasta, "linha_microfita_curvas.txt"),
        ],
    }
    modelos = [
        f"Branch-Line [{sufixo_sub}] (f_res S11) [GHz]",
        f"Cavidade [{sufixo_cav}] (TE_101) [GHz]",
        f"Antena Patch [{sufixo_sub}] (f_res S11) [GHz]",
        f"Divisor Wilkinson [{sufixo_sub}] (f_res S11) [GHz]",
        f"Linha de Transmissão [{sufixo_sub}] (min S11) [dB]",
    ]
    return pd.DataFrame(dados, index=modelos)


df_fr4 = montar_matriz_substrato(DIR_FR4, "FR-4", "Ar")
df_ro3010 = montar_matriz_substrato(DIR_RO3010, "RO3010", "Rutila")
df_todos = pd.concat([df_fr4, df_ro3010])

print("=== Matriz 1: FR-4 / Ar (5×3 = 15 elementos) ===")
display(df_fr4.round(4))

print("\n=== Matriz 2: Rogers RO3010 / Rutila (5×3 = 15 elementos) ===")
display(df_ro3010.round(4))

print("\n=== Matriz Consolidada: Todos os Modelos (10×3 = 30 elementos) ===")
display(df_todos.round(4))


=== Matriz 1: FR-4 / Ar (5×3 = 15 elementos) ===


,HFSS,Tidy3D,Analítico
Branch-Line [FR-4] (f_res S11) [GHz],2.9737,3.0000,3.0000
Cavidade [Ar] (TE_101) [GHz],3.6844,3.6855,3.6842
Antena Patch [FR-4] (f_res S11) [GHz],2.9944,2.9985,3.0000
Divisor Wilkinson [FR-4] (f_res S11) [GHz],3.0013,3.0000,3.0000
Linha de Transmissão [FR-4] (min S11) [dB],-40.3476,-37.9438,-66.3851



=== Matriz 2: Rogers RO3010 / Rutila (5×3 = 15 elementos) ===


,HFSS,Tidy3D,Analítico
Branch-Line [RO3010] (f_res S11) [GHz],3.0000,3.0000,3.0000
Cavidade [Rutila] (TE_101) [GHz],3.6844,3.6840,3.6842
Antena Patch [RO3010] (f_res S11) [GHz],3.0000,3.0045,3.0000
Divisor Wilkinson [RO3010] (f_res S11) [GHz],3.0013,3.0000,3.0000
Linha de Transmissão [RO3010] (min S11) [dB],-74.7871,-57.1652,-74.7831



=== Matriz Consolidada: Todos os Modelos (10×3 = 30 elementos) ===


,HFSS,Tidy3D,Analítico
Branch-Line [FR-4] (f_res S11) [GHz],2.9737,3.0000,3.0000
Cavidade [Ar] (TE_101) [GHz],3.6844,3.6855,3.6842
Antena Patch [FR-4] (f_res S11) [GHz],2.9944,2.9985,3.0000
Divisor Wilkinson [FR-4] (f_res S11) [GHz],3.0013,3.0000,3.0000
Linha de Transmissão [FR-4] (min S11) [dB],-40.3476,-37.9438,-66.3851
Branch-Line [RO3010] (f_res S11) [GHz],3.0000,3.0000,3.0000
Cavidade [Rutila] (TE_101) [GHz],3.6844,3.6840,3.6842
Antena Patch [RO3010] (f_res S11) [GHz],3.0000,3.0045,3.0000
Divisor Wilkinson [RO3010] (f_res S11) [GHz],3.0013,3.0000,3.0000
Linha de Transmissão [RO3010] (min S11) [dB],-74.7871,-57.1652,-74.7831


In [3]:
def gerar_anotacoes_e_desvio(df: pd.DataFrame, unidades: list[str]):
    df_desvio = df.sub(df["Analítico"], axis=0).div(df["Analítico"].abs(), axis=0) * 100.0
    anot = np.empty(df.shape, dtype=object)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            val = df.iloc[i, j]
            if j < 2:
                diff = df_desvio.iloc[i, j]
                anot[i, j] = f"{val:.4f} {unidades[i]}\n({diff:+.2f}%)"
            else:
                anot[i, j] = f"{val:.4f} {unidades[i]}\n(ref.)"
    return df_desvio, anot


unidades_5 = ["GHz", "GHz", "GHz", "GHz", "dB"]
desv_fr4, anot_fr4 = gerar_anotacoes_e_desvio(df_fr4, unidades_5)
desv_ro, anot_ro = gerar_anotacoes_e_desvio(df_ro3010, unidades_5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.2), dpi=140)

sns.heatmap(
    desv_fr4.abs(),
    annot=anot_fr4,
    fmt="",
    cmap="YlGnBu",
    linewidths=1.2,
    linecolor="white",
    cbar_kws={"label": "Desvio Absoluto vs. Analítico (%)"},
    ax=ax1,
)
ax1.set_title("Matriz 5×3 — Substrato FR-4 ($\varepsilon_r = 4{,}4$) / Cavidade a Ar", fontsize=11.5, fontweight="bold", pad=10)
ax1.set_xlabel("Método", fontweight="bold")
ax1.set_ylabel("Modelo / Métrica", fontweight="bold")
ax1.tick_params(axis="y", rotation=0)

sns.heatmap(
    desv_ro.abs(),
    annot=anot_ro,
    fmt="",
    cmap="YlOrRd",
    linewidths=1.2,
    linecolor="white",
    cbar_kws={"label": "Desvio Absoluto vs. Analítico (%)"},
    ax=ax2,
)
ax2.set_title("Matriz 5×3 — Substrato Rogers RO3010 ($\varepsilon_r = 10{,}2$) / Rutila ($\varepsilon_r = 82$)", fontsize=11.5, fontweight="bold", pad=10)
ax2.set_xlabel("Método", fontweight="bold")
ax2.set_ylabel("")
ax2.tick_params(axis="y", rotation=0)

plt.suptitle("Comparação das Matrizes (HFSS vs. Tidy3D vs. Analítico)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


ValueError: 
Matriz 5×3 — Substrato FR-4 ($arepsilon_r = 4{,}4$) / Cavidade a Ar
                             ^
ParseException: Expected end of text, found '$'  (at char 29), (line:1, col:30)

ValueError: 
Matriz 5×3 — Substrato FR-4 ($arepsilon_r = 4{,}4$) / Cavidade a Ar
                             ^
ParseException: Expected end of text, found '$'  (at char 29), (line:1, col:30)

<Figure size 2240x868 with 4 Axes>

In [ ]:
# Painel Comparativo em Barras para Todos os Modelos (FR-4/Ar vs. RO3010/Rutila)
fig, (ax_f, ax_db) = plt.subplots(1, 2, figsize=(15.5, 5.5), dpi=140, gridspec_kw={"width_ratios": [2.2, 1.0]})

# 1) Modelos de Ressonância / Modo em GHz (8 modelos: 4 FR-4/Ar + 4 RO3010/Rutila)
df_freq_all = pd.concat([df_fr4.iloc[:4], df_ro3010.iloc[:4]])
df_freq_all.index = [
    "Branch-Line (FR-4)",
    "Cavidade (Ar)",
    "Patch (FR-4)",
    "Wilkinson (FR-4)",
    "Branch-Line (RO3010)",
    "Cavidade (Rutila)",
    "Patch (RO3010)",
    "Wilkinson (RO3010)",
]
df_freq_all.plot(kind="bar", ax=ax_f, width=0.78, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax_f.set_title("Frequências de Ressonância / Modo TE101 — Todos os Modelos (GHz)", fontsize=11.5, fontweight="bold")
ax_f.set_ylabel("Frequência (GHz)")
ax_f.set_ylim(2.8, 3.85)
ax_f.tick_params(axis="x", rotation=25)
ax_f.legend(title="Método")

# 2) Linha de Transmissão (menor S11 em dB: FR-4 vs. RO3010)
df_db_all = pd.concat([df_fr4.iloc[[4]], df_ro3010.iloc[[4]]])
df_db_all.index = ["Linha Microfita (FR-4)", "Linha Microfita (RO3010)"]
df_db_all.plot(kind="bar", ax=ax_db, width=0.65, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax_db.set_title("Linha de Transmissão — Menor S11 (dB)", fontsize=11.5, fontweight="bold")
ax_db.set_ylabel("S11 mínimo (dB)")
ax_db.set_ylim(-85, 0)
ax_db.tick_params(axis="x", rotation=10)
ax_db.legend(title="Método", loc="lower right")

plt.tight_layout()
plt.show()
